<a href="https://colab.research.google.com/github/novikovamaria137-png/mtuci-llm-course/blob/main/lesson-2.1/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# Практика 2.1. Рабочее окружение LLM-разработчика

**Модуль 2 · Урок 1 · 110 минут**

К концу этой практики у вас будет собственный каркас проекта, который вы будете дополнять все десять уроков модуля. Ничего устанавливать заранее не нужно — ноутбук работает в Colab, локально и в любом Jupyter.

---

### Что вы сделаете

| Шаг | Что делаем | Время | Нужен ключ |
|---|---|---|---|
| 0 | Определим, где запущен ноутбук | 5 мин | нет |
| 1 | Установим зависимости с зафиксированными версиями | 10 мин | нет |
| 2 | Соберём структуру проекта | 15 мин | нет |
| 3 | Напишем модуль настроек — поиск ключа в трёх местах | 20 мин | нет |
| 4 | Напишем клиент: вызов, повтор с выдержкой, учёт расхода | 25 мин | нет |
| 5 | Прогоним шесть самопроверок каркаса | 15 мин | нет |
| 6 | Подключим ключ — или обойдёмся без него | 5 мин | да |
| 7 | Сделаем первый вызов и посчитаем расход | 5 мин | да |
| 8 | Защитимся от утечки ключа в репозиторий | 10 мин | нет |

> **Ключ не обязателен.** Если ключа у вас пока нет, всё будет работать в автономном режиме на заглушке. Это сделано намеренно: отсутствие доступа к API не должно останавливать обучение.

---
## Шаг 0. Где я запущен?

Первое, что должен уметь ваш код, — понимать, в какой среде он оказался. От этого зависит, откуда брать ключи и куда складывать файлы.

*Статус ячейки: проверено запуском.*

In [ ]:
import sys, platform, os
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

COLAB = in_colab()
print("Среда:          ", "Google Colab" if COLAB else "локальная среда / другой Jupyter")
print("Python:         ", sys.version.split()[0])
print("Система:        ", platform.system(), platform.machine())
print("Рабочая папка:  ", Path.cwd())

assert sys.version_info >= (3, 9), "Нужен Python 3.9 или новее"
print("\nПроверка версии Python пройдена.")

---
## Шаг 1. Зависимости

Ячейка ниже устойчива к повторному запуску: если пакет уже стоит, установка пропускается. Это важно — в Colab вы будете перезапускать ноутбук не раз.

**Почему версии зафиксированы.** Библиотеки для работы с LLM меняются быстро, и код, написанный полгода назад, может перестать работать после обновления. Фиксация версий делает вашу работу воспроизводимой — вы сможете вернуться к ней через месяц и получить тот же результат.

*Статус ячейки: требует проверки при запуске потока — состав пакетов может измениться.*

In [ ]:
REQUIREMENTS = [
    "openai==2.51.0",       # клиент к OpenAI-совместимым API
    "python-dotenv==1.2.2",  # чтение .env
]

import importlib.util, subprocess, sys

def ensure(spec):
    name = spec.split("==")[0].replace("-", "_")
    if importlib.util.find_spec(name) is None:
        print(f"  устанавливаю {spec} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
    else:
        print(f"  {name}: уже установлен")

print("Проверяю зависимости:")
for spec in REQUIREMENTS:
    ensure(spec)
print("\nГотово.")

---
## Шаг 2. Структура проекта

Мы не будем писать весь код в ноутбуке. Вместо этого соберём небольшой модуль `llmcourse`, который будет расти от урока к уроку. К концу модуля у вас окажется собственная библиотека — она же пойдёт в итоговый проект.

```
llm-project/
  llmcourse/
    __init__.py
    config.py      ← настройки и ключи   (шаг 3)
    client.py      ← работа с моделью    (шаг 4)
  .env             ← ваши ключи, В РЕПОЗИТОРИЙ НЕ ПОПАДАЕТ
  .env.example     ← шаблон без ключей, попадает
  .gitignore
  requirements.txt
```

**Почему не всё в ноутбуке.** Код в ноутбуке невозможно переиспользовать и трудно тестировать. Как только фрагмент понадобился дважды — он переезжает в модуль. Это тот же принцип, что и с промптом в модуле 1: хранить в одном месте, а не копировать.

> **Ловушка, на которую напарываются все.** Мы создаём файлы `.py` уже после того, как Python осмотрел содержимое папки. Он это запомнил — и нового файла «не увидит», выдав `ImportError`. Лечится строкой `importlib.invalidate_caches()`, и вы встретите её ниже дважды. Запомните симптом: файл на месте, а импорт не работает.

*Статус ячейки: проверено запуском.*

In [ ]:
from pathlib import Path

ROOT = Path("llm-project")
(ROOT / "llmcourse").mkdir(parents=True, exist_ok=True)
(ROOT / "llmcourse" / "__init__.py").write_text("", encoding="utf-8")

# чтобы импортировать модуль из ноутбука
import sys
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

print("Создана структура:")
for p in sorted(ROOT.rglob("*")):
    print("  ", p)

---
## Шаг 3. Модуль настроек

Здесь решается главный вопрос безопасности всего модуля: **где живут ключи**.

Правило одно и оно без исключений: **ключ никогда не пишется в коде и не попадает в репозиторий.** Ключ в коде — это не «пока так, потом поправлю». Это ключ, который уже утёк, потому что история коммитов помнит всё.

Наш модуль ищет ключ в трёх местах по порядку:

| Где | Когда используется |
|---|---|
| Colab Secrets | в Colab — значок ключа на левой панели |
| Переменные окружения | на сервере, в CI |
| Файл `.env` | при локальной работе |

Если ключа нет нигде — включается автономный режим. Практики продолжают работать на заглушке.

*Статус ячейки: проверено запуском.*

In [ ]:
%%writefile llm-project/llmcourse/config.py
"""Единая точка настройки. Урок 2.1.

Ключи НИКОГДА не пишутся в коде. Порядок поиска:
  1. Colab Secrets  (значок ключа слева в Colab)
  2. переменные окружения
  3. файл .env рядом с проектом
Если ключа нет — включается автономный режим на заглушке.
"""
import os
from pathlib import Path

ENV_KEYS = ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL")


def _from_colab(name):
    try:
        from google.colab import userdata          # есть только в Colab
        return userdata.get(name)
    except Exception:
        return None


def _from_dotenv(name, path=".env"):
    p = Path(path)
    if not p.exists():
        return None
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        if k.strip() == name:
            return v.strip().strip('"').strip("'")
    return None


def get(name, default=None):
    """Достаёт значение из Colab Secrets, окружения или .env."""
    return _from_colab(name) or os.environ.get(name) or _from_dotenv(name) or default


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def settings():
    """Возвращает конфигурацию и признак автономного режима."""
    cfg = {k: get(k) for k in ENV_KEYS}
    cfg["OFFLINE"] = not bool(cfg["LLM_API_KEY"])
    cfg["MODEL"] = cfg["LLM_MODEL"] or "demo-model"
    return cfg


def describe():
    """Человекочитаемый отчёт об окружении."""
    cfg = settings()
    where = "Google Colab" if in_colab() else "локальная среда"
    return "\n".join([
        f"Среда:            {where}",
        f"Модель:           {cfg['MODEL']}",
        f"Базовый адрес:    {cfg['LLM_BASE_URL'] or 'не задан'}",
        f"Ключ:             {'найден' if not cfg['OFFLINE'] else 'НЕ найден'}",
        f"Режим:            {'автономный (заглушка)' if cfg['OFFLINE'] else 'обращение к API'}",
    ])


In [ ]:
import importlib
importlib.invalidate_caches()   # файл создан уже после того, как Python осмотрел папку

from llmcourse import config
importlib.reload(config)        # чтобы правки подхватывались при повторном запуске

print(config.describe())

---
## Шаг 4. Клиент к модели

Теперь главный компонент. Три вещи, которые отличают рабочий клиент от учебного примера в три строки:

**1. Он написан на OpenAI-совместимый интерфейс.** Российские API и локальные рантаймы его поддерживают. Смена поставщика — правка двух строк в `.env`, код не меняется. Это не абстрактная забота: доступность сервисов меняется, и переписывать проект из-за этого не хочется.

**2. Он повторяет запрос при временных сбоях — но только их.** Превышен лимит запросов, таймаут, сбой на стороне сервиса — повторяем с нарастающей задержкой. Неверный запрос или отсутствующая модель — не повторяем: результат будет тот же, а время и деньги потратим.

**3. Он считает расход.** Каждое обращение стоит денег, и узнавать сколько из счёта в конце месяца — плохая идея.

Подробно про параметры, стриминг и коды ошибок — следующий урок. Сейчас нам нужен работающий каркас.

*Статус ячейки: проверено запуском (автономный режим и логика повторов). Обращение к реальному API требует проверки на живом ключе.*

In [ ]:
%%writefile llm-project/llmcourse/client.py
"""Клиент для работы с языковой моделью. Уроки 2.1–2.2.

Написан на OpenAI-совместимый интерфейс: работает с российскими API
и с локальными рантаймами. Смена поставщика — правка .env, не кода.
Без ключа работает в автономном режиме на заглушке.
"""
import time, random, hashlib
from dataclasses import dataclass
from . import config


@dataclass
class Usage:
    """Накопительный счётчик расхода."""
    calls: int = 0
    tokens_in: int = 0
    tokens_out: int = 0
    price_in: float = 0.0     # рублей за 1000 входных токенов
    price_out: float = 0.0    # рублей за 1000 выходных

    @property
    def cost(self):
        return (self.tokens_in / 1000 * self.price_in +
                self.tokens_out / 1000 * self.price_out)

    def report(self):
        return (f"обращений: {self.calls}   "
                f"токенов: {self.tokens_in} вход / {self.tokens_out} выход   "
                f"стоимость: {self.cost:.4f} руб.")


def approx_tokens(text):
    """Грубая ОЦЕНКА числа токенов до вызова API.

    Это оценка, а не замер: точное число даёт токенизатор конкретной
    модели (см. урок 1.2). Нужна, чтобы прикинуть стоимость заранее.
    """
    return max(1, len(text) // 3)


class LLM:
    def __init__(self, price_in=0.0, price_out=0.0, max_retries=4, timeout=60):
        cfg = config.settings()
        self.offline = cfg["OFFLINE"]
        self.model = cfg["MODEL"]
        self.base_url = cfg["LLM_BASE_URL"]
        self.max_retries = max_retries
        self.usage = Usage(price_in=price_in, price_out=price_out)
        self._client = None
        if not self.offline:
            from openai import OpenAI
            self._client = OpenAI(base_url=self.base_url,
                                  api_key=cfg["LLM_API_KEY"],
                                  timeout=timeout)

    def _offline_answer(self, messages):
        """Детерминированный ответ: одинаковый запрос — одинаковый ответ."""
        text = " ".join(m["content"] for m in messages)
        h = hashlib.sha256(text.encode()).hexdigest()[:6]
        return (f"[автономный режим] Ответ-заглушка {h}. "
                f"Получено сообщений: {len(messages)}, символов: {len(text)}. "
                f"Подставьте ключ, чтобы обратиться к модели.")

    @staticmethod
    def _is_retryable(e):
        """Повторяем только то, что имеет шанс пройти со второго раза."""
        if type(e).__name__ in ("RateLimitError", "APITimeoutError",
                                "APIConnectionError", "InternalServerError",
                                "TimeoutError", "ConnectionError"):
            return True
        return getattr(e, "status_code", None) in (408, 429, 500, 502, 503, 504)

    def _with_retry(self, fn):
        """Экспоненциальная задержка со случайной добавкой."""
        for attempt in range(self.max_retries):
            try:
                return fn()
            except Exception as e:
                if not self._is_retryable(e) or attempt == self.max_retries - 1:
                    raise
                time.sleep(0.5 * (2 ** attempt) + random.uniform(0, 0.3))

    def ask(self, prompt, system=None, temperature=0.2, max_tokens=None):
        messages = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        self.usage.calls += 1

        if self.offline:
            answer = self._offline_answer(messages)
            self.usage.tokens_in += approx_tokens(" ".join(m["content"] for m in messages))
            self.usage.tokens_out += approx_tokens(answer)
            return answer

        def call():
            kw = dict(model=self.model, messages=messages, temperature=temperature)
            if max_tokens:
                kw["max_tokens"] = max_tokens
            return self._client.chat.completions.create(**kw)

        resp = self._with_retry(call)
        u = getattr(resp, "usage", None)
        if u:
            self.usage.tokens_in += getattr(u, "prompt_tokens", 0)
            self.usage.tokens_out += getattr(u, "completion_tokens", 0)
        return resp.choices[0].message.content


---
## Шаг 5. Проверяем каркас

Прежде чем подключать ключ, убедимся, что каркас работает. Проверки ниже — это, по сути, первые тесты вашего проекта.

Обратите внимание на последнюю проверку: **ошибка «неверный запрос» не повторяется.** Это не мелочь. Клиент, который повторяет любую ошибку, при неправильном запросе потратит четыре попытки и время ожидания — и всё равно упадёт.

*Статус ячейки: проверено запуском.*

In [ ]:
import importlib
importlib.invalidate_caches()        # см. пояснение выше — без этой строки будет ImportError

import llmcourse.client
importlib.reload(llmcourse.client)   # подхватываем правки при повторном запуске
from llmcourse.client import LLM, Usage, approx_tokens

llm = LLM(price_in=0.20, price_out=0.60)   # цены подставьте свои
print("Режим:", "автономный" if llm.offline else "обращение к API")

# 1. Заглушка детерминирована
a1, a2 = llm.ask("Привет", system="Ты помощник"), llm.ask("Привет", system="Ты помощник")
assert a1 == a2, "одинаковый запрос должен давать одинаковый ответ"
assert a1 != llm.ask("Другой запрос")
print("[ok] заглушка детерминирована")

# 2. Оценка токенов
assert approx_tokens("") == 1 and approx_tokens("а" * 300) == 100
print("[ok] оценка токенов")

# 3. Расчёт стоимости
u = Usage(tokens_in=2000, tokens_out=500, price_in=0.20, price_out=0.60)
assert abs(u.cost - 0.70) < 1e-9
print(f"[ok] расчёт стоимости: {u.cost:.2f} руб. за 2000+500 токенов")

# 4. Какие ошибки повторяем
class RateLimitError(Exception): pass
class BadRequestError(Exception): pass
assert LLM._is_retryable(RateLimitError())
assert not LLM._is_retryable(BadRequestError())
print("[ok] лимит запросов повторяем, неверный запрос — нет")

# 5. Повторы действительно работают
n = {"i": 0}
def flaky():
    n["i"] += 1
    if n["i"] < 3:
        raise ConnectionError("временный сбой")
    return "готово"
assert llm._with_retry(flaky) == "готово" and n["i"] == 3
print(f"[ok] восстановился после {n['i'] - 1} сбоёв подряд")

# 6. Неповторяемая ошибка не повторяется
m = {"i": 0}
def bad():
    m["i"] += 1
    raise BadRequestError("неверный запрос")
try:
    llm._with_retry(bad)
except BadRequestError:
    pass
assert m["i"] == 1, "неповторяемую ошибку повторять нельзя"
print("[ok] неверный запрос не повторяется — попытка одна")

print("\nКаркас работает.")

---
## Шаг 6. Подключаем ключ

Теперь подключим настоящий доступ. **Если ключа нет — пропустите этот шаг**, всё остальное будет работать в автономном режиме.

### В Google Colab

1. Слева на панели — значок ключа («Secrets»).
2. Добавьте три записи: `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.
3. Напротив каждой включите доступ для этого ноутбука.

Colab Secrets хранятся в вашем аккаунте, а не в ноутбуке. Ноутбуком можно делиться — ключи не уедут вместе с ним.

### Локально

Создайте файл `.env` рядом с проектом. Ячейка ниже создаст шаблон.

> ⚠️ **Что бы вы ни делали — не вставляйте ключ прямо в ячейку ноутбука.** Он сохранится в файле, попадёт в репозиторий и останется в истории коммитов навсегда. Это самая частая утечка в учебных проектах.

*Статус ячейки: проверено запуском.*

In [ ]:
from pathlib import Path

EXAMPLE = """# Шаблон настроек. Скопируйте в .env и подставьте свои значения.
# Файл .env в репозиторий НЕ попадает — он в .gitignore.

LLM_BASE_URL=https://адрес-вашего-поставщика/v1
LLM_API_KEY=сюда-ваш-ключ
LLM_MODEL=имя-модели
"""

(ROOT / ".env.example").write_text(EXAMPLE, encoding="utf-8")
print("Создан llm-project/.env.example\n")
print(EXAMPLE)

env_path = ROOT / ".env"
if env_path.exists():
    print("Файл .env уже существует — оставляю как есть.")
else:
    print("Файла .env пока нет. Создайте его на основе .env.example,")
    print("если у вас есть ключ. Без ключа практика работает в автономном режиме.")

---
## Шаг 7. Первый вызов и учёт расхода

*Статус ячейки: проверено запуском в автономном режиме. Обращение к реальному API требует проверки на живом ключе.*

In [ ]:
llm = LLM(price_in=0.20, price_out=0.60)   # подставьте тарифы своего поставщика

ответ = llm.ask(
    prompt="Перечисли три причины, по которым языковая модель может ошибиться.",
    system="Ты помощник инженера. Отвечай коротко и по существу.",
    temperature=0.2,
)

print("ОТВЕТ:\n")
print(ответ)
print("\n" + "-" * 60)
print("РАСХОД:", llm.usage.report())

if llm.offline:
    print("\nЭто автономный режим. Ответ — заглушка, расход посчитан по оценке.")
    print("Подставьте ключ и запустите ячейку заново, чтобы обратиться к модели.")

### Оценка стоимости до отправки

Полезная привычка: прикинуть расход **до** того, как отправить в модель длинный текст. Особенно когда в цикле обрабатываются сотни документов.

*Статус ячейки: проверено запуском.*

In [ ]:
ДОКУМЕНТ = "Текст документа. " * 500          # ~8500 символов
ОТВЕТ_ПРИМЕРНО = 400                          # ожидаемая длина ответа в токенах

вход = approx_tokens(ДОКУМЕНТ)
цена = вход / 1000 * 0.20 + ОТВЕТ_ПРИМЕРНО / 1000 * 0.60

print(f"Оценка входа:        {вход} токенов")
print(f"Ожидаемый выход:     {ОТВЕТ_ПРИМЕРНО} токенов")
print(f"Стоимость обращения: {цена:.4f} руб.")
print(f"На 1000 документов:  {цена * 1000:.2f} руб.")
print("\nЭто оценка. Точное число токенов даёт токенизатор модели — урок 1.2.")

---
## Шаг 8. Защита от утечки ключа

Последний и самый важный шаг практики. Создадим `.gitignore` и проверим, что ключ не попадёт в репозиторий.

**Почему это отдельный шаг урока.** Утёкший ключ — самая частая и самая дорогая ошибка начинающих. Публичные репозитории регулярно сканируются, и найденный ключ используют в течение минут. Удаление файла следующим коммитом не помогает: старая версия остаётся в истории.

*Статус ячейки: проверено запуском.*

In [ ]:
GITIGNORE = """# Секреты — никогда не в репозиторий
.env
*.key
secrets/

# Python
__pycache__/
*.py[cod]
.venv/
venv/

# Данные и артефакты
data/raw/
*.index
.ipynb_checkpoints/
"""

(ROOT / ".gitignore").write_text(GITIGNORE, encoding="utf-8")
(ROOT / "requirements.txt").write_text("\n".join(REQUIREMENTS) + "\n", encoding="utf-8")
print("Созданы .gitignore и requirements.txt\n")

# --- проверка на утечку ---
import re

ПОДОЗРИТЕЛЬНОЕ = [
    (re.compile(r"(?i)(api[_-]?key|token|secret)\s*=\s*['\"][^'\"]{12,}"), "ключ прямо в коде"),
    (re.compile(r"sk-[A-Za-z0-9]{16,}"), "ключ формата sk-..."),
    (re.compile(r"(?i)Bearer\s+[A-Za-z0-9._-]{20,}"), "токен в заголовке"),
]

ИСКЛЮЧЕНИЯ = {".env", ".env.example"}

def проверить(корень):
    находки = []
    for f in Path(корень).rglob("*"):
        if not f.is_file() or f.name in ИСКЛЮЧЕНИЯ:
            continue
        if f.suffix not in {".py", ".ipynb", ".txt", ".md", ".yaml", ".yml", ".json"}:
            continue
        try:
            текст = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for шаблон, описание in ПОДОЗРИТЕЛЬНОЕ:
            if шаблон.search(текст):
                находки.append((f, описание))
    return находки

находки = проверить(ROOT)
if находки:
    print("НАЙДЕНЫ ВОЗМОЖНЫЕ УТЕЧКИ:")
    for f, описание in находки:
        print(f"   {f}: {описание}")
else:
    print("[ok] Подозрительных мест не найдено.")

# проверка работоспособности самого детектора
(ROOT / "_проверка.py").write_text('API_KEY = "abcdefghijklmnop123456"\n', encoding="utf-8")
assert проверить(ROOT), "детектор обязан был сработать"
(ROOT / "_проверка.py").unlink()
print("[ok] Детектор проверен на подставном файле и сработал.")

---
## Задание

### Обязательная часть

1. **Добавьте в `config.py` функцию `require(name)`**, которая возвращает значение или выбрасывает понятную ошибку с подсказкой, где это значение задать. Сейчас `get()` молча возвращает `None`, и ошибка всплывает позже и не там.

2. **Добавьте в `LLM` метод `ask_many(prompts)`** — обработку списка запросов с накоплением расхода. Предусмотрите, что один из запросов может упасть: остальные должны обработаться.

3. **Замерьте стоимость своей задачи из модуля 1.** Возьмите проектную задачу, оцените размер типового запроса и ответа, посчитайте месячный расход.

4. **Инициализируйте репозиторий** и сделайте первый коммит. Убедитесь, что `.env` в него не попал:
   ```
   git init && git add . && git status
   ```
   Файла `.env` в списке быть не должно.

### Часть повышенной сложности

5. **Добавьте кэширование.** Если тот же запрос с теми же параметрами уже задавался — верните сохранённый ответ, не обращаясь к API. Это заметно экономит на отладке, когда вы гоняете один и тот же набор примеров.

   Подумайте: что именно должно входить в ключ кэша? Достаточно ли текста запроса?

6. **Добавьте предел расходов.** Пусть клиент выбрасывает исключение, если накопленная стоимость превысила заданный порог. Защита от цикла, который случайно ушёл в тысячу обращений.

---

## Чек-лист

- [ ] Ноутбук выполняется целиком без ошибок
- [ ] Структура проекта создана, `llmcourse` импортируется
- [ ] `config.describe()` показывает корректную среду
- [ ] Все шесть проверок каркаса пройдены
- [ ] `.gitignore` создан, `.env` в него входит
- [ ] Проверка на утечку выполнена, находок нет
- [ ] Ключа нет в ячейках ноутбука
- [ ] Реализованы `require()` и `ask_many()`
- [ ] Посчитана стоимость своей задачи
- [ ] Сделан первый коммит, `.env` в него не попал

---

## Частые проблемы

| Симптом | Причина | Что делать |
|---|---|---|
| `ModuleNotFoundError: llmcourse` | Папка проекта не в `sys.path` | Перезапустите шаг 2 |
| Изменения в `.py` не подхватываются | Python кэширует модули | `importlib.reload()` или перезапуск среды |
| В Colab не виден ключ | Не включён доступ для ноутбука | Значок ключа слева, переключатель напротив записи |
| `.env` не читается | Файл не в рабочей папке | Проверьте `Path.cwd()` в шаге 0 |
| Всё работает, но ответы одинаковые | Автономный режим | Так и должно быть без ключа |

---

## Что дальше

Каркас готов. В уроке 2.2 мы разберём то, что сегодня осталось за кадром: параметры запроса, стриминг ответа, коды ошибок и работу с лимитами. Клиент, который вы написали, будет дополняться — и к концу модуля станет основой вашего итогового проекта.